# 13 · Theory test: predict the self-scoring failure temperature from human text only (design doc §8.8.18, pre-registered)

**How to run**: pick a Colab server with an **A100 GPU** (an L4 also works, more slowly) and click "Run All". Rough estimate, not measured: about 2 hours on an A100. If the connection drops, click "Run All" again; finished parts are skipped. Results go to `DATA_DIR/13_theory_prediction_test/`.

**What it computes**: for four base models from new families (OLMo-2 1B, Granite-3.3 2B, GPT-Neo 1.3B, BLOOM 1.7B), each model first scores the 1,000 held-out M4GT human texts. From these human scores alone it predicts where its own Fast-DetectGPT score fails: T* = 1 - E_h[mean(H - s)] / mean(Var[log p]). The predictions are written to `predictions.json` **before any text is generated**. Only then does each model continue 300 prompts at ten fixed temperatures and score its own continuations, and the notebook compares the observed AUROC = 0.5 crossing with the prediction and with two simple baselines (T* = 1 and the development mean 1.011).

In [ ]:
# 1. Check for a GPU with bfloat16 support (A100 or L4).
import torch
assert torch.cuda.is_available(), "No GPU: switch the Colab server to an A100 GPU"
name = torch.cuda.get_device_name(0); mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(name, f"{mem:.0f} GB")
assert torch.cuda.is_bf16_supported(), f"{name} has no bfloat16 support; choose an A100 or L4 GPU"
assert mem > 15, f"This {name} has only {mem:.0f} GB; choose an A100 or L4 GPU"

In [ ]:
# 2. Mount Google Drive and switch to the data folder.
import os
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/curvature-margin/data"  # folder with the layout in data/README.md
%cd $DATA_DIR
OUT = "13_theory_prediction_test"
os.makedirs(OUT, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf"

In [ ]:
# 3. Install packages (as in 09).
%pip install -q -U transformers huggingface-hub accelerate
import os, json, gc, glob, shutil, time, hashlib, sys, platform
from datetime import datetime, timezone
import numpy as np, pandas as pd
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# 4. Held-out M4GT human texts (as in 09): check the raw file and manifest hashes, then every text hash.
RAW = "external/m4gt/SubtaskB.jsonl"; MAN = "manifests/m4gt_humans_A_manifest.csv"
EXPECTED_RAW = "4d65fc1fb93b5c9d21b226aca06b610ee031fa9ccbc18fa6f39d16273dd281c7"
EXPECTED_MAN = "7c6f2b6529a8ef7b1a1920bc798d45c68a7a7162823566f52a73deb6ae8d1caf"

def sha256_file(path):
    d = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            d.update(block)
    return d.hexdigest()

assert sha256_file(RAW) == EXPECTED_RAW, "Raw file SHA-256 does not match; stopping."
assert sha256_file(MAN) == EXPECTED_MAN, "Human manifest SHA-256 does not match; stopping."
man = pd.read_csv(MAN)
assert len(man) == 1000 and man.id.tolist() == list(range(1000)) and (man.role[:300] == "source").all()
need = {int(r.line_number): r for r in man.itertuples(index=False)}
found = {}
with open(RAW, encoding="utf-8") as f:
    for ln, line in enumerate(f, start=1):
        r = need.get(ln)
        if r is None:
            continue
        it = json.loads(line)
        assert hashlib.sha256(it["text"].encode("utf-8")).hexdigest() == r.text_sha256, f"Text hash mismatch on line {ln}"
        assert it["source"] == r.source and it["model"] == "human"
        found[int(r.id)] = dict(id=int(r.id), domain=r.source, text=it["text"])
assert len(found) == 1000
humans = [found[i] for i in range(1000)]; sources = humans[:300]
print(len(humans), "human texts; prompts:", len(sources))

In [ ]:
# 5. Pre-registered settings (design doc 8.8.18) and shared functions. Continuation rules as in 06 and 09; scoring as in 06 (four rows).
MODELS = {"allenai/OLMo-2-0425-1B": "a1847dff35000b4271fa70afc5db10fd29fedbdf",
          "ibm-granite/granite-3.3-2b-base": "c15dc681f4b2aeb4e576cdaae5434edfc17ed14d",
          "EleutherAI/gpt-neo-1.3B": "dbe59a7f4a88d01d1ba9798d78dbe3fe038792c8",
          "bigscience/bloom-1b7": "cc72a88036c2fb937d65efeacc57a0c2ef5d6fe5"}
TEMPS = [0.92, 0.94, 0.96, 0.98, 1.0, 1.02, 1.04, 1.06, 1.08, 1.14]
PREFIX, MAX_TOK, BS, SBS, SEED, DEV_MEAN = 30, 512, 32, 16, 42, 1.011
tag_of = lambda name: name.split("/")[1]

def load(name):
    tok = AutoTokenizer.from_pretrained(name, revision=MODELS[name])
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(name, revision=MODELS[name], dtype=torch.bfloat16).cuda().eval()
    return tok, model

def free(name):
    gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(f"/content/hf/hub/models--{name.replace('/', '--')}", ignore_errors=True)

def bos(tok):   # the BOS id if this tokenizer adds one by default (none of the development models did)
    first = tok("a")["input_ids"][:1]
    return first if tok.bos_token_id is not None and first == [tok.bos_token_id] else []

def prepare(tok, texts):   # split on raw text tokens; a default BOS is added again when generating and scoring
    items = []
    for h in texts:
        ids = tok(h["text"], add_special_tokens=False, truncation=True, max_length=MAX_TOK - 1)["input_ids"]
        if len(ids) < PREFIX + 20:
            continue
        items.append(dict(id=h["id"], domain=h["domain"], prefix_ids=ids[:PREFIX], target=len(ids) - PREFIX,
                          human=tok.decode(ids[PREFIX:], clean_up_tokenization_spaces=False),
                          prefix=tok.decode(ids[:PREFIX], clean_up_tokenization_spaces=False)))
    return items

@torch.no_grad()
def score_self(tok, model, recs, name):
    tok.padding_side = "right"; pad = tok.pad_token_id
    enc = []
    for r in recs:
        k = len(tok(r["prefix"])["input_ids"])
        ids = tok(r["prefix"] + r["text"], truncation=True, max_length=MAX_TOK)["input_ids"]
        if len(ids) - k >= 10:
            enc.append((r, k, ids))
    enc.sort(key=lambda e: len(e[2])); out = []
    for s in range(0, len(enc), SBS):
        chunk = enc[s:s + SBS]; L = max(len(e[2]) for e in chunk)
        x = torch.full((len(chunk), L), pad); att = torch.zeros((len(chunk), L), dtype=torch.long)
        for i, (_, _, ids) in enumerate(chunk):
            x[i, :len(ids)] = torch.tensor(ids); att[i, :len(ids)] = 1
        logits = model(input_ids=x.cuda(), attention_mask=att.cuda()).logits
        for i, (r, k, ids) in enumerate(chunk):
            n = len(ids)
            lp = torch.log_softmax(logits[i, :n - 1].float(), -1)
            tgt = torch.tensor(ids[1:], device=lp.device)
            p = lp.exp(); e1 = (p * lp).sum(-1)
            arr = torch.stack([-lp.gather(1, tgt[:, None])[:, 0], -e1, (p * lp * lp).sum(-1) - e1 ** 2,
                               p.topk(10, -1).values.sum(-1)]).cpu().numpy().astype(np.float32)[:, k - 1:]
            out.append(dict(id=r["id"], domain=r["domain"], model=name, decoding=r["decoding"], who=r["who"], scorer=name, arr=arr))
    return pd.DataFrame(out)

@torch.no_grad()
def generate_batch(model, tok, batch, temp, special):
    tok.padding_side = "left"; pad = tok.pad_token_id; B = bos(tok)
    L = max(len(B) + len(b["prefix_ids"]) for b in batch); new = max(b["target"] for b in batch)
    x = torch.full((len(batch), L), pad); att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        ids = B + b["prefix_ids"]
        x[i, L - len(ids):] = torch.tensor(ids); att[i, L - len(ids):] = 1
    out = model.generate(input_ids=x.cuda(), attention_mask=att.cuda(), do_sample=True, temperature=temp, top_p=1.0, top_k=0,
                         max_new_tokens=new, min_new_tokens=new, suppress_tokens=special, pad_token_id=pad, repetition_penalty=1.0)
    gen = out[:, L:].cpu()
    res = [gen[i, :b["target"]].tolist() for i, b in enumerate(batch)]
    for b, ids in zip(batch, res):   # self-check: exact length, no special tokens
        assert len(ids) == b["target"], ("wrong length", len(ids), b["target"])
        assert not (set(ids) & set(special)), "a special token was generated"
    return res

def curvature_stats(frame):
    c = np.array([(a[1] - a[0]).mean() for a in frame.arr]); V = np.array([a[2].mean() for a in frame.arr])
    return c, V

In [ ]:
# 6. Stage 1, BEFORE any generation: each model scores the human texts; the prediction uses these human scores only.
preds_path = f"{OUT}/predictions.json"
preds = json.load(open(preds_path)) if os.path.exists(preds_path) else {}
for name in MODELS:
    tag = tag_of(name); hpath = f"{OUT}/humans_self_{tag}.pkl"
    if not os.path.exists(hpath):
        t0 = time.time(); tok, model = load(name)
        items = prepare(tok, humans)
        recs = [dict(id=b["id"], domain=b["domain"], decoding="human", who="human", prefix=b["prefix"], text=b["human"]) for b in items]
        score_self(tok, model, recs, name).to_pickle(hpath)
        del model; free(name); print(f"{tag}: scored {len(recs)} human texts, {(time.time() - t0) / 60:.1f} min", flush=True)
    c, V = curvature_stats(pd.read_pickle(hpath))
    value = dict(model=name, revision=MODELS[name], n_humans=int(len(c)), E_h_c=float(c.mean()), V_human=float(V.mean()),
                 T_pred=float(1 - c.mean() / V.mean()))
    if tag in preds:   # a rerun must reproduce the stored prediction exactly
        assert abs(preds[tag]["T_pred"] - value["T_pred"]) < 1e-9, f"{tag}: stored prediction differs from the recomputed one"
    else:
        preds[tag] = dict(value, written_at_utc=datetime.now(timezone.utc).isoformat())
        json.dump(preds, open(preds_path, "w"), indent=2)
    print(f"{tag}: E_h[c] = {value['E_h_c']:+.4f}, V = {value['V_human']:.4f}  ->  predicted failure temperature {value['T_pred']:.4f}")
PRED_SHA = sha256_file(preds_path)
print("predictions.json SHA-256:", PRED_SHA)

In [ ]:
# 6b. Corrected predictions (design doc 8.8.20), from human text only: include the tokens banned during generation
#     (special and end-of-text tokens). Must finish before stage 2 generates any further text.
corr_path = f"{OUT}/predictions_corrected.json"
corr = json.load(open(corr_path)) if os.path.exists(corr_path) else {}

@torch.no_grad()
def banned_stats(tok, model, recs, banned):
    tok.padding_side = "right"; pad = tok.pad_token_id
    enc = []
    for r in recs:
        k = len(tok(r["prefix"])["input_ids"])
        ids = tok(r["prefix"] + r["text"], truncation=True, max_length=MAX_TOK)["input_ids"]
        if len(ids) - k >= 10:
            enc.append((k, ids))
    enc.sort(key=lambda e: len(e[1])); delta, var = [], []
    for s in range(0, len(enc), SBS):
        chunk = enc[s:s + SBS]; L = max(len(e[1]) for e in chunk)
        x = torch.full((len(chunk), L), pad); att = torch.zeros((len(chunk), L), dtype=torch.long)
        for i, (_, ids) in enumerate(chunk):
            x[i, :len(ids)] = torch.tensor(ids); att[i, :len(ids)] = 1
        logits = model(input_ids=x.cuda(), attention_mask=att.cuda()).logits
        for i, (k, ids) in enumerate(chunk):
            lp = torch.log_softmax(logits[i, :len(ids) - 1].float(), -1)[k - 1:]   # continuation positions, as in score_self
            p = lp.exp(); H = -(p * lp).sum(-1)
            keep = torch.ones(lp.shape[-1], device=lp.device); keep[torch.tensor(banned, device=lp.device)] = 0
            pm = p * keep; Z = pm.sum(-1)
            m1 = (pm * lp).sum(-1) / Z; m2 = (pm * lp * lp).sum(-1) / Z               # moments of log p under the renormalized p'
            delta.append(float((H + m1).mean())); var.append(float((m2 - m1 ** 2).mean()))
    return np.array(delta), np.array(var)

for name in MODELS:
    tag = tag_of(name)
    c, _ = curvature_stats(pd.read_pickle(f"{OUT}/humans_self_{tag}.pkl"))
    if tag not in corr:
        t0 = time.time(); tok, model = load(name)
        banned = sorted(set(tok.all_special_ids) | {tok.eos_token_id} | set(np.atleast_1d(model.generation_config.eos_token_id).tolist() if model.generation_config.eos_token_id is not None else []))
        items = prepare(tok, humans)
        d, v = banned_stats(tok, model, [dict(prefix=b["prefix"], text=b["human"]) for b in items], banned)
        del model; free(name)
        corr[tag] = dict(model=name, revision=MODELS[name], banned_ids=[int(b) for b in banned], n_humans=int(len(d)), delta=float(d.mean()),
                         V_restricted=float(v.mean()), E_h_c=float(c.mean()), T_pred_corrected=float(1 + (d.mean() - c.mean()) / v.mean()),
                         written_at_utc=datetime.now(timezone.utc).isoformat())
        json.dump(corr, open(corr_path, "w"), indent=2)
        print(f"{tag}: {len(d)} human texts, {(time.time() - t0) / 60:.1f} min", flush=True)
    print(f"{tag}: original prediction {preds[tag]['T_pred']:.4f} | corrected prediction {corr[tag]['T_pred_corrected']:.4f} (delta {corr[tag]['delta']:+.4f})")
CORR_SHA = sha256_file(corr_path)
print("predictions_corrected.json SHA-256:", CORR_SHA)

In [ ]:
# 7. Stage 2: continuations at the ten fixed temperatures (300 prompts each), then each model scores its own continuations.
assert os.path.exists(preds_path) and len(json.load(open(preds_path))) == len(MODELS), "Stage 1 predictions are missing; run cell 6 first"
assert os.path.exists(corr_path) and len(json.load(open(corr_path))) == len(MODELS), "Corrected predictions are missing; run cell 6b first"
for name in MODELS:
    tag = tag_of(name); spath = f"{OUT}/scores_self_{tag}.pkl"
    todo = [t for t in TEMPS if not os.path.exists(f"{OUT}/gen_{tag}_T{t}.json")]
    if os.path.exists(spath) and not todo:
        print("Already done, skipping:", spath); continue
    tok, model = load(name)
    special = sorted(set(tok.all_special_ids) | {tok.eos_token_id} | set(np.atleast_1d(model.generation_config.eos_token_id).tolist() if model.generation_config.eos_token_id is not None else []))
    items = sorted(prepare(tok, sources), key=lambda b: b["target"])
    assert len(items) == 300, f"Only {len(items)} prompts left; some texts are too short"
    for temp in todo:
        torch.manual_seed(SEED); t0 = time.time(); rows = []
        for s in range(0, len(items), BS):
            batch = items[s:s + BS]
            for b, ids in zip(batch, generate_batch(model, tok, batch, temp, special)):
                rows.append(dict(id=b["id"], domain=b["domain"], model=name, decoding=f"T{temp}", prefix=b["prefix"],
                                 ai=tok.decode(ids, clean_up_tokenization_spaces=False), n_tokens=len(ids)))
        json.dump(rows, open(f"{OUT}/gen_{tag}_T{temp}.json", "w"), ensure_ascii=False)
        print(f"{tag} T={temp}: {len(rows)} texts, {(time.time() - t0) / 60:.1f} min", flush=True)
    recs = [dict(id=r["id"], domain=r["domain"], decoding=r["decoding"], who="ai", prefix=r["prefix"], text=r["ai"])
            for t in TEMPS for r in json.load(open(f"{OUT}/gen_{tag}_T{t}.json"))]
    ai = score_self(tok, model, recs, name)
    pd.concat([pd.read_pickle(f"{OUT}/humans_self_{tag}.pkl"), ai], ignore_index=True).to_pickle(spath)
    print(f"{tag}: scored {len(ai)} continuations -> {spath}", flush=True)
    del model; free(name)

In [ ]:
# 8. Result against the pre-registered criteria. The formal analysis is scripts/e9_theory_prediction.py.
from sklearn.metrics import roc_auc_score

def cross(x, y, level=0.0):
    x, y = np.asarray(x, float), np.asarray(y, float) - level
    for i in range(len(x) - 1):
        if y[i] > 0 >= y[i + 1] or y[i] < 0 <= y[i + 1]:
            return float(x[i] + (x[i + 1] - x[i]) * y[i] / (y[i] - y[i + 1]))
    return None

rows = []
for name in MODELS:
    tag = tag_of(name); s = pd.read_pickle(f"{OUT}/scores_self_{tag}.pkl")
    s["fdg"] = [(a[1] - a[0]).sum() / np.sqrt(a[2].sum()) for a in s.arr]
    h = s[s.who == "human"]; auc = []
    for t in TEMPS:
        g = s[s.decoding == f"T{t}"]
        auc.append(roc_auc_score(np.r_[np.ones(len(g)), np.zeros(len(h))], np.r_[g.fdg.values, h.fdg.values]))
    obs = cross(TEMPS, auc, 0.5); pred = preds[tag]["T_pred"]; predc = corr[tag]["T_pred_corrected"]
    rows.append(dict(model=tag, T_pred=pred, T_pred_corrected=predc, T_obs=obs, error=None if obs is None else obs - pred,
                     error_corrected=None if obs is None else obs - predc,
                     error_T1=None if obs is None else obs - 1.0, error_dev_mean=None if obs is None else obs - DEV_MEAN,
                     **{f"AUROC_T{t}": a for t, a in zip(TEMPS, auc)}))
R = pd.DataFrame(rows); R.to_csv(f"{OUT}/summary.csv", index=False)
print(R[["model", "T_pred", "T_pred_corrected", "T_obs", "error", "error_corrected", "error_T1", "error_dev_mean"]].round(4).to_string(index=False))
ok = R.dropna(subset=["error"])
if len(ok) < len(R):
    print("Undetermined (crossing outside the grid):", R[R.error.isna()].model.tolist())
mae, mae1, maed = ok.error.abs().mean(), ok.error_T1.abs().mean(), ok.error_dev_mean.abs().mean()
print(f"MAE: theory {mae:.4f} | corrected {ok.error_corrected.abs().mean():.4f} | guess T*=1 {mae1:.4f} | development mean 1.011 {maed:.4f}")
print("Criterion 1 (each |error| <= 0.02 and MAE <= 0.01):", "met" if (ok.error.abs() <= 0.02).all() and mae <= 0.01 else "not met")
print("Criterion 2 (MAE below both baselines):", "met" if mae < mae1 and mae < maed else "not met")

In [ ]:
# 9. Record the run and flush unsynced files back to Google Drive.
run_info = dict(completed_at_utc=datetime.now(timezone.utc).isoformat(), predictions_sha256=PRED_SHA, corrected_predictions_sha256=CORR_SHA, models=MODELS, temperatures=TEMPS,
                prefix_tokens=PREFIX, max_tokens=MAX_TOK, gen_seed=SEED, humans_manifest_sha256=EXPECTED_MAN, python=sys.version,
                platform=platform.platform(), torch=torch.__version__, transformers=transformers.__version__, gpu=torch.cuda.get_device_name(0))
json.dump(run_info, open(f"{OUT}/run_info.json", "w"), indent=2)
%cd /content
drive.flush_and_unmount()
print("Done: results are in DATA_DIR/13_theory_prediction_test/")